# AttackPath-PGAS — Reproducible Implementation

**Full-row IIoT streaming → hierarchical rare-event risk → PGAS latent attack-state inference → posterior uncertainty → validation-governed alerts.**

This notebook is distributed with **no saved outputs**. Configure the dataset path in Section 2 and run from top to bottom. Test labels are first consumed only at the final held-out evaluation boundary.

## 1. Imports and optional dependencies

In [ ]:
from __future__ import annotations
import os, sys, re, json, math, time, hashlib, heapq, platform, warnings
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict
from typing import Any, Iterable, Sequence

import numpy as np
import pandas as pd
import scipy
from scipy.special import expit, logit, logsumexp
from scipy.optimize import minimize
from scipy.stats import invgamma
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, matthews_corrcoef, roc_auc_score, average_precision_score, confusion_matrix,
    log_loss, brier_score_loss)
import joblib, psutil, arviz as az

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE=True
except Exception as exc:
    xgb=None; XGBOOST_AVAILABLE=False; XGBOOST_IMPORT_ERROR=repr(exc)

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 120)

## 2. Central configuration

In [ ]:
CFG = {
 'project': {'name':'AttackPath-PGAS','global_seed':42,'artifacts_dir':'artifacts'},
 'data': {'root':os.getenv('ATTACKPATH_DATA_ROOT','data/CICAPT-IIoT'),'phase1_files':[],'phase2_files':[]},
 'stream': {'chunk_rows':200_000,'schema_sample_rows':20_000},
 'window': {'rows':5_000,'attack_fraction_threshold':0.001,'min_attack_events':3,'include_final_partial_window':True,
            'max_numeric_columns':65,'max_event_type_levels':30,'min_feature_variance':1e-10,'max_core_features':90},
 'temporal': {'lags':[1,2,3,5],'rolling_windows':[3,5,9],'ewma_spans':[3,7],'base_feature_limit':35},
 'split': {'phase2_train_fraction':0.50,'phase2_val_fraction':0.25,'phase2_test_fraction':0.25},
 'features': {'mi_features':35,'shift_candidates':45,'scaler':'standard'},
 'event': {'max_numeric_features':45,'train_attack_cap':350_000,'train_benign_cap':650_000,
           'val_attack_cap':120_000,'val_benign_cap':240_000,'sampling_seed':2026,'top_risk_events':20,
           'high_risk_thresholds':[0.50,0.70,0.90,0.95], 'logistic_max_iter':300,'extra_trees':500,
           'hist_iter':450,'hist_lr':0.05,'xgb_trees':650,'xgb_depth':6,'xgb_lr':0.035,'xgb_subsample':0.90,'xgb_colsample':0.90},
 'window_models': {'logistic_l2_C':0.35,'logistic_l1_C':0.15,'logistic_max_iter':5000,'rf_trees':700,'et_trees':800,
                   'hgb_iter':600,'hgb_lr':0.035,'hgb_leaves':31,'hgb_l2':0.02,'xgb_trees':700,'xgb_depth':4,
                   'xgb_lr':0.025,'xgb_subsample':0.85,'xgb_colsample':0.85,'stack_logistic_C':0.20,
                   'stack_et_trees':600,'stack_rf_trees':500,'stack_hgb_iter':350,'stack_hgb_lr':0.04,'stack_hgb_leaves':15},
 'calibration': {'epsilon':1e-4},
 'hierarchical': {'threshold_grid':301,'min_recall':0.65,'max_fpr':0.05},
 'pgas': {'states':2,'chains':2,'particles':200,'iterations':550,'burn_in':180,'thin':5,
          'initial_prior':[300.,2.],'transition_prior_benign':[140.,4.],'transition_prior_attack':[6.,42.],
          'nig_m0':0.,'nig_kappa0':0.05,'nig_a0':3.,'nig_b0':3.,'risk_emission_weight':9.,
          'gaussian_emission_weight':0.005,'variance_floor':1e-6,'uncertainty_draws':80,'chain_seeds':[1042,2042]},
 'policy': {'linear_thresholds':401,'linear_range':[0.001,0.999],'empirical_quantiles':151,'min_recall':0.65,
            'max_fpr':0.05,'min_precision':0.20,'hysteresis_low_ratios':[0.50,0.55],
            'min_segment_lengths':[1,2,3],'merge_gaps':[0,1,2],
            'risk_anchor_weights':[0.80,0.85,0.90,0.95,0.97,0.99],
            'transition_boost':0.35,'guarded_pgas_noisy_or':0.15,'pgas_structural_bonus':0.015},
 'robustness': {'seeds':[42,123,202,777,999],'run':False},
 'external': {'run':False,'edge_iiot_root':'','ton_iot_root':''}
}
EPS=CFG['calibration']['epsilon']
ART=Path(CFG['project']['artifacts_dir'])
for sub in ['config','manifests','data','features','models','posterior','predictions','metrics']:
    (ART/sub).mkdir(parents=True,exist_ok=True)

## 3. Reproducibility manifest and SHA-256 helpers

In [ ]:
def set_seed(seed):
    np.random.seed(seed); os.environ['PYTHONHASHSEED']=str(seed)

def sha256_file(path, block=2**20):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(block), b''): h.update(b)
    return h.hexdigest()

def environment_manifest():
    d={'python':sys.version,'platform':platform.platform(),'machine':platform.machine(),'cpu_logical':os.cpu_count(),
       'ram_gb':round(psutil.virtual_memory().total/1024**3,3),'numpy':np.__version__,'pandas':pd.__version__,
       'scipy':scipy.__version__,'scikit_learn':sklearn.__version__,'arviz':az.__version__,
       'xgboost_available':XGBOOST_AVAILABLE}
    if XGBOOST_AVAILABLE: d['xgboost']=xgb.__version__
    return d

set_seed(CFG['project']['global_seed'])
config_path=ART/'config'/'config.json'
config_path.write_text(json.dumps(CFG,indent=2),encoding='utf-8')
(ART/'config'/'config.sha256').write_text(sha256_file(config_path)+'\n',encoding='utf-8')
(ART/'manifests'/'environment.json').write_text(json.dumps(environment_manifest(),indent=2),encoding='utf-8')

## 4. Dataset discovery, chunk readers and phase ownership

In [ ]:
def supported(p):
    s=p.name.lower(); return s.endswith('.csv') or s.endswith('.csv.gz') or s.endswith('.parquet')

def phase_marker(p):
    s=str(p).lower().replace('_','').replace('-','').replace(' ','')
    if 'phase1' in s or 'phaseone' in s: return 1
    if 'phase2' in s or 'phasetwo' in s: return 2
    return None

def discover_phase_files():
    if CFG['data']['phase1_files'] or CFG['data']['phase2_files']:
        p1=[Path(x) for x in CFG['data']['phase1_files']]; p2=[Path(x) for x in CFG['data']['phase2_files']]
    else:
        root=Path(CFG['data']['root'])
        if not root.exists(): raise FileNotFoundError(f'Dataset root not found: {root.resolve()}')
        files=sorted(p for p in root.rglob('*') if p.is_file() and supported(p))
        p1=[p for p in files if phase_marker(p)==1]; p2=[p for p in files if phase_marker(p)==2]
    if not p1 or not p2: raise RuntimeError('Resolve both phases explicitly in CFG if auto-discovery fails.')
    return p1,p2

def iter_file_chunks(path, chunk_rows):
    if path.name.lower().endswith('.parquet'):
        import pyarrow.parquet as pq
        pf=pq.ParquetFile(path)
        for rg in range(pf.num_row_groups):
            d=pf.read_row_group(rg).to_pandas()
            for start in range(0,len(d),chunk_rows): yield d.iloc[start:start+chunk_rows].copy()
    else:
        yield from pd.read_csv(path,chunksize=chunk_rows,low_memory=False)

def iter_phase_chunks(files, phase):
    phase_offset=0
    for order,p in enumerate(files):
        file_offset=0
        for chunk in iter_file_chunks(p,CFG['stream']['chunk_rows']):
            meta={'phase':phase,'file_order':order,'path':str(p),'phase_row_start':phase_offset,'file_row_start':file_offset}
            yield chunk,meta
            n=len(chunk); phase_offset+=n; file_offset+=n

def dataset_manifest(p1,p2):
    rows=[]
    for phase,files in [(1,p1),(2,p2)]:
        for order,p in enumerate(files):
            rows.append({'phase':phase,'file_order':order,'path':str(p.resolve()),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
    d=pd.DataFrame(rows); d.to_csv(ART/'manifests'/'dataset_manifest.csv',index=False); return d

## 5. Schema detection and binary label mapping

In [ ]:
ROLE_ALIASES={
 'src':['src_ip','source_ip','srcip','source','src_addr','source_address','ip_src'],
 'dst':['dst_ip','destination_ip','dstip','destination','dst_addr','destination_address','ip_dst'],
 'event':['event_type','eventtype','type','protocol','proto','activity','operation'],
 'pid':['process_id','processid','pid','process','process_name'],
 'label':['label','class','attack','target','binary_label','attack_label','category'],
 'time':['timestamp','time','datetime','date_time','ts']}

def ncol(c): return re.sub(r'[^a-z0-9]+','',str(c).lower())
def detect_role(cols,aliases):
    m={ncol(c):c for c in cols}
    for a in aliases:
        if ncol(a) in m: return m[ncol(a)]
    return None

def read_sample(path,n=20_000):
    return pd.read_parquet(path).head(n) if path.name.lower().endswith('.parquet') else pd.read_csv(path,nrows=n,low_memory=False)

def detect_schema(df):
    s={r:detect_role(df.columns,a) for r,a in ROLE_ALIASES.items()}
    if s['label'] is None: raise ValueError('Label column not detected; update ROLE_ALIASES.')
    return s

def binary_attack_label(series):
    num=pd.to_numeric(series,errors='coerce')
    if num.notna().mean()>0.95: return (~num.fillna(0).eq(0)).astype(np.int8).to_numpy()
    s=series.astype(str).str.strip().str.lower()
    benign={'0','0.0','benign','normal','normal.','false','background','no attack','non-attack','nonattack'}
    return (~s.isin(benign)).astype(np.int8).to_numpy()

## 6. Full-row 5,000-event window construction

In [ ]:
def entropy_counts(s):
    c=s.value_counts().to_numpy(float); c=c[c>0]
    if not len(c): return 0.
    p=c/c.sum(); return float(-(p*np.log(np.clip(p,1e-15,1))).sum())

def numeric_candidates(sample,schema,cap):
    excluded={v for v in schema.values() if v is not None}
    out=[]
    for c in sample.columns:
        if c in excluded: continue
        x=pd.to_numeric(sample[c],errors='coerce')
        if x.notna().mean()>=.5: out.append(c)
    return out[:cap]

def summarize_window(df,phase,w,schema,numcols,event_levels):
    y=binary_attack_label(df[schema['label']]); nattack=int(y.sum()); frac=nattack/max(len(df),1)
    out={'phase':phase,'phase_window':w,'n_events':len(df),'attack_event_count':nattack,'attack_event_fraction':frac,
         'window_label':int(nattack>=CFG['window']['min_attack_events'] or frac>=CFG['window']['attack_fraction_threshold'])}
    src,dst=schema.get('src'),schema.get('dst')
    if src and src in df:
        out['unique_src']=int(df[src].nunique(dropna=True)); out['src_entropy']=entropy_counts(df[src].astype(str))
    else: out['unique_src']=0; out['src_entropy']=0.
    if dst and dst in df:
        out['unique_dst']=int(df[dst].nunique(dropna=True)); out['dst_entropy']=entropy_counts(df[dst].astype(str))
    else: out['unique_dst']=0; out['dst_entropy']=0.
    if src and dst and src in df and dst in df:
        pairs=df[[src,dst]].astype(str); nodes=pd.unique(pd.concat([pairs[src],pairs[dst]],ignore_index=True)); edges=pairs.drop_duplicates()
        deg=pd.concat([pairs[src],pairs[dst]],ignore_index=True).value_counts(); n=len(nodes); e=len(edges)
        out.update(n_nodes=n,n_edges=e,directed_density=(e/(n*(n-1)) if n>=2 else 0.),degree_mean=float(deg.mean()),
                   degree_std=float(deg.std(ddof=0) if len(deg)>1 else 0.),degree_max=float(deg.max() if len(deg) else 0.))
    ev=schema.get('event')
    if ev and ev in df:
        se=df[ev].astype(str); out['event_entropy']=entropy_counts(se); vc=se.value_counts()
        for level in event_levels: out[f'event_count__{level}']=int(vc.get(level,0))
    pid=schema.get('pid'); out['n_processes']=int(df[pid].nunique(dropna=True)) if pid and pid in df else 0
    for c in numcols:
        if c not in df: continue
        x=pd.to_numeric(df[c],errors='coerce').replace([np.inf,-np.inf],np.nan).dropna().to_numpy(float)
        out[f'num__{c}__mean']=float(x.mean()) if len(x) else np.nan
        out[f'num__{c}__std']=float(x.std()) if len(x) else np.nan
        out[f'num__{c}__max']=float(x.max()) if len(x) else np.nan
    return out

def stream_build_windows(files,phase):
    sample=read_sample(files[0],CFG['stream']['schema_sample_rows']); schema=detect_schema(sample)
    numcols=numeric_candidates(sample,schema,CFG['window']['max_numeric_columns'])
    ev=schema.get('event'); levels=(sample[ev].astype(str).value_counts().head(CFG['window']['max_event_type_levels']).index.tolist() if ev else [])
    carry=pd.DataFrame(); rows=[]; w=0; W=CFG['window']['rows']
    for chunk,_ in iter_phase_chunks(files,phase):
        if not carry.empty: chunk=pd.concat([carry,chunk],ignore_index=True); carry=pd.DataFrame()
        while len(chunk)>=W:
            rows.append(summarize_window(chunk.iloc[:W].copy(),phase,w,schema,numcols,levels)); w+=1; chunk=chunk.iloc[W:].reset_index(drop=True)
        carry=chunk
    if len(carry) and CFG['window']['include_final_partial_window']: rows.append(summarize_window(carry,phase,w,schema,numcols,levels))
    return pd.DataFrame(rows),{'schema':schema,'numeric_cols':numcols,'event_levels':levels}

## 7. Chronological split and no-peeking feature screen

In [ ]:
META={'global_window','phase','phase_window','window_label','attack_event_count','attack_event_fraction'}
def attach_global(w1,w2):
    d=pd.concat([w1,w2],ignore_index=True); d.insert(0,'global_window',np.arange(len(d))); return d

def split_indices(w1,w2):
    n2=len(w2); nt=int(round(.50*n2)); nv=int(round(.25*n2)); g2=np.arange(len(w1),len(w1)+n2)
    return {'train':np.r_[np.arange(len(w1)),g2[:nt]],'val':g2[nt:nt+nv],'test':g2[nt+nv:],
            'phase2_train_local':np.arange(nt),'phase2_val_local':np.arange(nt,nt+nv),'phase2_test_local':np.arange(nt+nv,n2)}

def std_shift(a,b):
    a=pd.to_numeric(a,errors='coerce').dropna(); b=pd.to_numeric(b,errors='coerce').dropna()
    if len(a)<2 or len(b)<2: return 0.
    return float(abs(a.mean()-b.mean())/math.sqrt((a.var()+b.var())/2+1e-8))

def feature_screen(d,split):
    cand=[c for c in d.columns if c not in META]; X=d[cand].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    var=X.iloc[split['train']].var(); cand=var[var>CFG['window']['min_feature_variance']].index.tolist()
    p1=d.index[d.phase.eq(1)].to_numpy(); p2tr=split['train'][d.iloc[split['train']].phase.to_numpy()==2]
    shifts={c:std_shift(X.loc[p1,c],X.loc[p2tr,c]) for c in cand}
    shift=[c for c,_ in sorted(shifts.items(),key=lambda kv:kv[1],reverse=True)][:CFG['features']['shift_candidates']]
    imp=SimpleImputer(strategy='median'); Xtr=imp.fit_transform(X.loc[split['train'],cand]); y=d.loc[split['train'],'window_label'].to_numpy(int)
    mi=mutual_info_classif(Xtr,y,random_state=CFG['project']['global_seed']); mi_order=[c for _,c in sorted(zip(mi,cand),reverse=True)]
    mif=mi_order[:CFG['features']['mi_features']]; core=[]
    for c in mif+shift:
        if c not in core: core.append(c)
    return {'mi_features':mif,'shift_candidates':shift,'core_features':core[:CFG['window']['max_core_features']]}

def add_temporal(d,base):
    out=d.copy(); base=list(base)[:CFG['temporal']['base_feature_limit']]
    for _,idx in out.groupby('phase',sort=False).groups.items():
        idx=np.asarray(list(idx)); block=out.loc[idx,base].apply(pd.to_numeric,errors='coerce')
        for c in base:
            s=block[c]
            for lag in CFG['temporal']['lags']:
                out.loc[idx,f'{c}__lag{lag}']=s.shift(lag).to_numpy(); out.loc[idx,f'{c}__diff{lag}']=(s-s.shift(lag)).to_numpy()
            for w in CFG['temporal']['rolling_windows']:
                r=s.rolling(w,min_periods=1); out.loc[idx,f'{c}__rmean{w}']=r.mean().to_numpy(); out.loc[idx,f'{c}__rstd{w}']=r.std(ddof=0).fillna(0).to_numpy(); out.loc[idx,f'{c}__rmax{w}']=r.max().to_numpy()
            for span in CFG['temporal']['ewma_spans']: out.loc[idx,f'{c}__ewma{span}']=s.ewm(span=span,adjust=False).mean().to_numpy()
    return out

## 8. Calibration, model metrics and documented validation scores

In [ ]:
def clip_prob(p): return np.clip(np.asarray(p,float),EPS,1-EPS)
@dataclass
class AffineLogitCalibrator:
    a:float=1.; b:float=0.
    def fit(self,p,y):
        z=logit(clip_prob(p)); y=np.asarray(y,int)
        def obj(t): return log_loss(y,clip_prob(expit(t[0]*z+t[1])),labels=[0,1])
        r=minimize(obj,[1.,0.],method='L-BFGS-B')
        if r.success: self.a,self.b=map(float,r.x)
        return self
    def predict(self,p): return clip_prob(expit(self.a*logit(clip_prob(p))+self.b))

def calibration_errors(y,p,bins=15):
    y=np.asarray(y,int); p=clip_prob(p); edges=np.linspace(0,1,bins+1); ids=np.minimum(np.digitize(p,edges[1:-1]),bins-1)
    ece=mce=0.; rows=[]
    for b in range(bins):
        m=ids==b
        if not m.any(): continue
        conf=float(p[m].mean()); rate=float(y[m].mean()); gap=abs(conf-rate); ece+=m.mean()*gap; mce=max(mce,gap)
        rows.append({'bin':b,'n':int(m.sum()),'mean_probability':conf,'attack_rate':rate,'abs_gap':gap})
    return float(ece),float(mce),pd.DataFrame(rows)

def threshold_metrics(y,p,t):
    pred=(clip_prob(p)>=t).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'f1':f1_score(y,pred,zero_division=0),'f2':fbeta_score(y,pred,beta=2,zero_division=0),
            'mcc':matthews_corrcoef(y,pred) if len(np.unique(pred))>1 else 0.,'recall':recall_score(y,pred,zero_division=0),
            'precision':precision_score(y,pred,zero_division=0),'fpr':fp/max(fp+tn,1)}

def best_operating(y,p,n=301):
    best=None
    for t in np.linspace(.001,.999,n):
        m=threshold_metrics(y,p,t); row={'threshold':t,**m}
        if best is None or m['f2']>best['f2']: best=row
    return best

def event_quality(y,p):
    b=best_operating(y,p); aupr=average_precision_score(y,p); return {'aupr':aupr,'best_f2':b['f2'],'score':aupr+.25*b['f2']}

def window_quality(y,p):
    b=best_operating(y,p); aupr=average_precision_score(y,p); auroc=roc_auc_score(y,p)
    return {'aupr':aupr,'auroc':auroc,'best_f2':b['f2'],'best_mcc':b['mcc'],'quality':.45*aupr+.30*b['f2']+.15*b['mcc']+.10*auroc}

## 9. Event sampling, learners and full-stream risk aggregation

In [ ]:
def event_features(sample,schema):
    bad={v for v in schema.values() if v is not None}; out=[]
    for c in sample.columns:
        if c in bad or any(t in ncol(c) for t in ['label','class','attack','target','time','process','pid']): continue
        if pd.to_numeric(sample[c],errors='coerce').notna().mean()>=.5: out.append(c)
    return out[:CFG['event']['max_numeric_features']]

def bounded(existing,incoming,cap,rng):
    if incoming.empty: return existing
    d=pd.concat([existing,incoming],ignore_index=True)
    if len(d)<=cap: return d
    ids=rng.choice(len(d),cap,replace=False); return d.iloc[np.sort(ids)].reset_index(drop=True)

def stream_samples(files,phase,schema,features,split):
    rng=np.random.default_rng(CFG['event']['sampling_seed']+phase); buckets={k:pd.DataFrame() for k in ['train_attack','train_benign','val_attack','val_benign']}; W=CFG['window']['rows']
    for chunk,meta in iter_phase_chunks(files,phase):
        for c in features:
            if c not in chunk: chunk[c]=np.nan
        y=binary_attack_label(chunk[schema['label']]); rows=np.arange(meta['phase_row_start'],meta['phase_row_start']+len(chunk)); wins=rows//W
        owner=np.full(len(chunk),'ignore',object)
        if phase==1: owner[:] = 'train'
        else:
            owner[np.isin(wins,split['phase2_train_local'])]='train'; owner[np.isin(wins,split['phase2_val_local'])]='val'
        frame=chunk[features].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan); frame['_y']=y; frame['_owner']=owner
        for own in ['train','val']:
            for cls,name in [(1,'attack'),(0,'benign')]:
                part=frame[(frame._owner==own)&(frame._y==cls)].drop(columns=['_owner']); key=f'{own}_{name}'; cap=CFG['event'][f'{own}_{name}_cap']
                buckets[key]=bounded(buckets[key],part,cap,rng)
    return {'train':pd.concat([buckets['train_attack'],buckets['train_benign']],ignore_index=True),
            'val':pd.concat([buckets['val_attack'],buckets['val_benign']],ignore_index=True)}

def make_event_models(seed):
    d={'EventLogisticBalanced':LogisticRegression(class_weight='balanced',max_iter=CFG['event']['logistic_max_iter'],random_state=seed),
       'EventExtraTreesBalanced':ExtraTreesClassifier(n_estimators=CFG['event']['extra_trees'],class_weight='balanced_subsample',n_jobs=-1,random_state=seed),
       'EventHistGradientBoostingWeighted':HistGradientBoostingClassifier(max_iter=CFG['event']['hist_iter'],learning_rate=CFG['event']['hist_lr'],random_state=seed)}
    if XGBOOST_AVAILABLE:
        d['EventXGBoostWeighted']=xgb.XGBClassifier(n_estimators=CFG['event']['xgb_trees'],max_depth=CFG['event']['xgb_depth'],learning_rate=CFG['event']['xgb_lr'],subsample=.90,colsample_bytree=.90,objective='binary:logistic',eval_metric='logloss',n_jobs=-1,random_state=seed)
    return d

def class_weights(y):
    n0=max((y==0).sum(),1); n1=max((y==1).sum(),1); return np.where(y==1,n0/n1,1.)

def fit_event_ensemble(samples,features,seed=42):
    tr,va=samples['train'],samples['val']; ytr=tr['_y'].to_numpy(int); yv=va['_y'].to_numpy(int)
    imp=SimpleImputer(strategy='median'); sc=StandardScaler(); Xtr=sc.fit_transform(imp.fit_transform(tr[features])); Xv=sc.transform(imp.transform(va[features])); sw=class_weights(ytr)
    models={}; cals={}; qual={}
    for name,m in make_event_models(seed).items():
        try:
            if isinstance(m,HistGradientBoostingClassifier) or name.startswith('EventXGBoost'): m.fit(Xtr,ytr,sample_weight=sw)
            else: m.fit(Xtr,ytr)
            cal=AffineLogitCalibrator().fit(m.predict_proba(Xv)[:,1],yv); p=cal.predict(m.predict_proba(Xv)[:,1]); models[name]=m; cals[name]=cal; qual[name]=event_quality(yv,p)
        except Exception as exc: qual[name]={'unavailable':True,'error':repr(exc)}
    names=list(models); s=np.array([max(qual[n]['score'],EPS) for n in names]); w=s/s.sum()
    return {'models':models,'calibrators':cals,'qualities':qual,'weights':dict(zip(names,w)),'imputer':imp,'scaler':sc,'features':list(features)}

def event_predict(bundle,frame):
    X=bundle['scaler'].transform(bundle['imputer'].transform(frame[bundle['features']])); p=np.zeros(len(frame))
    for n,m in bundle['models'].items(): p+=bundle['weights'][n]*bundle['calibrators'][n].predict(m.predict_proba(X)[:,1])
    return clip_prob(p)

@dataclass
class RiskAcc:
    count:int=0; sum_:float=0.; sumsq:float=0.; max_:float=0.; logsurv:float=0.; top:list|None=None; exceed:dict|None=None
    def __post_init__(self):
        self.top=[] if self.top is None else self.top; self.exceed={t:0 for t in CFG['event']['high_risk_thresholds']} if self.exceed is None else self.exceed
    def add(self,p):
        p=clip_prob(p); self.count+=len(p); self.sum_+=p.sum(); self.sumsq+=(p*p).sum(); self.max_=max(self.max_,float(p.max(initial=0))); self.logsurv+=np.log1p(-p).sum()
        for t in self.exceed: self.exceed[t]+=int((p>=t).sum())
        k=CFG['event']['top_risk_events']; local=p if len(p)<=k else p[np.argpartition(p,-k)[-k:]]
        for v in local:
            if len(self.top)<k: heapq.heappush(self.top,float(v))
            elif v>self.top[0]: heapq.heapreplace(self.top,float(v))
    def done(self):
        n=max(self.count,1); mean=self.sum_/n; var=max(self.sumsq/n-mean*mean,0.)
        d={'event_risk_mean':mean,'event_risk_std':math.sqrt(var),'event_risk_max':self.max_,'event_risk_top20_mean':float(np.mean(self.top)) if self.top else 0.,'event_risk_noisy_or':1-math.exp(min(self.logsurv,0.))}
        for t,c in self.exceed.items(): d[f'event_risk_ge_{str(t).replace(".","_")}']=c/n
        return d

def aggregate_full_event_risk(files,phase,bundle):
    acc=defaultdict(RiskAcc); W=CFG['window']['rows']
    for chunk,meta in iter_phase_chunks(files,phase):
        for c in bundle['features']:
            if c not in chunk: chunk[c]=np.nan
        p=event_predict(bundle,chunk[bundle['features']].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)); wins=np.arange(meta['phase_row_start'],meta['phase_row_start']+len(chunk))//W
        for w in np.unique(wins): acc[int(w)].add(p[wins==w])
    return pd.DataFrame([{'phase':phase,'phase_window':w,**a.done()} for w,a in sorted(acc.items())])

## 10. Frozen window preprocessor and candidate models

In [ ]:
@dataclass
class FrozenPreprocessor:
    cols:list; imputer:Any; scaler:Any
    def transform(self,d):
        X=d.reindex(columns=self.cols).apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan); return self.scaler.transform(self.imputer.transform(X))

def fit_preprocessor(d,train,cols):
    X=d.loc[train,cols].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan); imp=SimpleImputer(strategy='median'); sc=StandardScaler() if CFG['features']['scaler']=='standard' else RobustScaler(); sc.fit(imp.fit_transform(X)); return FrozenPreprocessor(list(cols),imp,sc)

class PCARecon:
    def __init__(self): self.pca=PCA(n_components=.95,svd_solver='full')
    def fit(self,X): self.pca.fit(X); return self
    def score(self,X):
        Z=self.pca.transform(X); R=self.pca.inverse_transform(Z); return np.mean((X-R)**2,axis=1)

def make_window_models(seed):
    c=CFG['window_models']; d={
      'LogisticRegressionL2':LogisticRegression(C=c['logistic_l2_C'],class_weight='balanced',max_iter=c['logistic_max_iter'],solver='liblinear',random_state=seed),
      'LogisticRegressionL1':LogisticRegression(C=c['logistic_l1_C'],penalty='l1',class_weight='balanced',max_iter=c['logistic_max_iter'],solver='liblinear',random_state=seed),
      'RandomForestBalanced':RandomForestClassifier(n_estimators=c['rf_trees'],class_weight='balanced_subsample',max_features='sqrt',n_jobs=-1,random_state=seed),
      'ExtraTreesBalanced':ExtraTreesClassifier(n_estimators=c['et_trees'],class_weight='balanced_subsample',max_features='sqrt',n_jobs=-1,random_state=seed),
      'HistGradientBoostingWeighted':HistGradientBoostingClassifier(max_iter=c['hgb_iter'],learning_rate=c['hgb_lr'],max_leaf_nodes=c['hgb_leaves'],l2_regularization=c['hgb_l2'],random_state=seed)}
    if XGBOOST_AVAILABLE: d['XGBoostWeighted']=xgb.XGBClassifier(n_estimators=c['xgb_trees'],max_depth=c['xgb_depth'],learning_rate=c['xgb_lr'],subsample=c['xgb_subsample'],colsample_bytree=c['xgb_colsample'],objective='binary:logistic',eval_metric='logloss',n_jobs=-1,random_state=seed)
    return d

def fit_window_models(X,y,split,window_df,seed=42):
    out={'models':{},'cals':{},'qualities':{},'probs':{'train':{},'val':{},'test':{}}}; tr,va=split['train'],split['val']; sw=class_weights(y[tr])
    for name,m in make_window_models(seed).items():
        try:
            if isinstance(m,HistGradientBoostingClassifier) or name.startswith('XGBoost'): m.fit(X[tr],y[tr],sample_weight=sw)
            else: m.fit(X[tr],y[tr])
            cal=AffineLogitCalibrator().fit(m.predict_proba(X[va])[:,1],y[va]); out['models'][name]=m; out['cals'][name]=cal
            for s,idx in [('train',tr),('val',va),('test',split['test'])]: out['probs'][s][name]=cal.predict(m.predict_proba(X[idx])[:,1])
            out['qualities'][name]=window_quality(y[va],out['probs']['val'][name])
        except Exception as exc: out['qualities'][name]={'unavailable':True,'error':repr(exc)}
    p1=np.intersect1d(tr,window_df.index[window_df.phase.eq(1)].to_numpy())
    if len(p1)>10:
        iso=IsolationForest(n_estimators=400,random_state=seed,n_jobs=-1).fit(X[p1]); pca=PCARecon().fit(X[p1])
        for name,score,obj in [('IsolationForestBackground',lambda Z:-iso.score_samples(Z),iso),('PCABackgroundError',pca.score,pca)]:
            rv=score(X[va]); lo,hi=np.percentile(rv,[1,99]); pv0=clip_prob((rv-lo)/max(hi-lo,1e-12)); cal=AffineLogitCalibrator().fit(pv0,y[va]); out['models'][name]=obj; out['cals'][name]=(cal,lo,hi)
            for s,idx in [('train',tr),('val',va),('test',split['test'])]: out['probs'][s][name]=cal.predict(clip_prob((score(X[idx])-lo)/max(hi-lo,1e-12)))
            out['qualities'][name]=window_quality(y[va],out['probs']['val'][name])
    return out

## 11. Score stacking and hierarchical risk selection

In [ ]:
def score_features(p):
    s=pd.Series(clip_prob(p)); return pd.DataFrame({'p':s,'logit':logit(clip_prob(s)),'rank':s.rank(pct=True),'rmean3':s.rolling(3,min_periods=1).mean(),'rmean7':s.rolling(7,min_periods=1).mean(),'rmax5':s.rolling(5,min_periods=1).max(),'rmax11':s.rolling(11,min_periods=1).max(),'ewma5':s.ewm(span=5,adjust=False).mean(),'diff1':s.diff().fillna(0)}).to_numpy()

def stackers(bundle,y,split,seed=42):
    names=[n for n,q in bundle['qualities'].items() if not q.get('unavailable',False)]; Z={s:np.concatenate([score_features(bundle['probs'][s][n]) for n in names],axis=1) for s in ['train','val','test']}; c=CFG['window_models']; sw=class_weights(y[split['train']])
    models={'ScoreStackLogistic':LogisticRegression(C=c['stack_logistic_C'],class_weight='balanced',max_iter=5000,solver='liblinear',random_state=seed),
            'ScoreStackExtraTrees':ExtraTreesClassifier(n_estimators=c['stack_et_trees'],class_weight='balanced_subsample',n_jobs=-1,random_state=seed),
            'ScoreStackRandomForest':RandomForestClassifier(n_estimators=c['stack_rf_trees'],class_weight='balanced_subsample',n_jobs=-1,random_state=seed),
            'ScoreStackHGB':HistGradientBoostingClassifier(max_iter=c['stack_hgb_iter'],learning_rate=c['stack_hgb_lr'],max_leaf_nodes=c['stack_hgb_leaves'],random_state=seed)}
    for name,m in models.items():
        if isinstance(m,HistGradientBoostingClassifier): m.fit(Z['train'],y[split['train']],sample_weight=sw)
        else: m.fit(Z['train'],y[split['train']])
        cal=AffineLogitCalibrator().fit(m.predict_proba(Z['val'])[:,1],y[split['val']]); bundle['models'][name]=m; bundle['cals'][name]=cal
        for s in Z: bundle['probs'][s][name]=cal.predict(m.predict_proba(Z[s])[:,1])
        bundle['qualities'][name]=window_quality(y[split['val']],bundle['probs']['val'][name])
    return bundle

def fuse_base(probs,qualities):
    names=[n for n in probs if n in qualities and not qualities[n].get('unavailable',False)]; P=np.column_stack([clip_prob(probs[n]) for n in names]); q=np.array([max(qualities[n]['quality'],.01) for n in names]); q=q/q.sum()
    return {'quality_weighted':P@q,'mean':P.mean(1),'max':P.max(1),'top2_mean':np.sort(P,axis=1)[:,-min(2,P.shape[1]):].mean(1),'noisy_or':1-np.prod(1-P,axis=1)}

def refine(p):
    s=pd.Series(clip_prob(p)); return {'raw':clip_prob(s),'rollmean5':clip_prob(.8*s+.2*s.rolling(5,min_periods=1).mean()),'rollmax7':clip_prob(.85*s+.15*s.rolling(7,min_periods=1).max()),'ewma5':clip_prob(.8*s+.2*s.ewm(span=5,adjust=False).mean())}

def select_hierarchical(bundle,y,split):
    B={s:fuse_base(bundle['probs'][s],bundle['qualities']) for s in ['train','val','test']}; best=None; fallback=[]
    for b in B['val']:
        R={s:refine(B[s][b]) for s in B}
        for r in R['val']:
            cal=AffineLogitCalibrator().fit(R['val'][r],y[split['val']]); pv=cal.predict(R['val'][r]); ptr=cal.predict(R['train'][r]); pte=cal.predict(R['test'][r])
            for t in np.linspace(.001,.999,CFG['hierarchical']['threshold_grid']):
                m=threshold_metrics(y[split['val']],pv,t); J=m['f1']+.25*m['f2']+.20*m['mcc']+.25*m['recall']-.20*m['fpr']+.10*average_precision_score(y[split['val']],pv); feasible=m['recall']>=.65 and m['fpr']<=.05
                row={'base':b,'refinement':r,'threshold':float(t),'calibrator':cal,'J':J,'metrics':m,'train_p':ptr,'val_p':pv,'test_p':pte,'feasible':feasible}; fallback.append(row)
                if feasible and (best is None or J>best['J']): best=row
    return best if best is not None else max(fallback,key=lambda d:d['J'])

## 12. Hybrid emission, conditional SMC and PGAS

In [ ]:
def log_gauss_diag(X,mu,var):
    var=np.maximum(var,CFG['pgas']['variance_floor']); return -.5*(np.log(2*np.pi*var).sum()+((X-mu)**2/var).sum(1))
def log_emission(X,risk,theta):
    r=clip_prob(risk); out=np.empty((len(X),2)); lg=CFG['pgas']['gaussian_emission_weight']; lr=CFG['pgas']['risk_emission_weight']
    for k in [0,1]: out[:,k]=lg*log_gauss_diag(X,theta['mu'][k],theta['var'][k])+lr*np.log(r if k else 1-r)
    return out

def init_theta(X,risk,rng):
    pi=rng.dirichlet(CFG['pgas']['initial_prior']); A=np.vstack([rng.dirichlet(CFG['pgas']['transition_prior_benign']),rng.dirichlet(CFG['pgas']['transition_prior_attack'])]); z=(risk>=np.quantile(risk,.98)).astype(int); mu=np.zeros((2,X.shape[1])); var=np.ones_like(mu)
    for k in [0,1]:
        b=X[z==k]
        if len(b): mu[k]=b.mean(0); var[k]=np.maximum(b.var(0),1e-3)
    return {'pi':pi,'A':A,'mu':mu,'var':var}

def conditional_smc(X,risk,theta,ref,N,rng):
    T=len(X); le=log_emission(X,risk,theta); z=np.zeros((T,N),np.int8); anc=np.zeros((T,N),np.int32); lw=np.zeros((T,N))
    z[0,:N-1]=rng.choice(2,N-1,p=theta['pi']); z[0,N-1]=ref[0]; lw[0]=le[0,z[0]]; W=np.exp(lw[0]-logsumexp(lw[0])); ll=logsumexp(lw[0])-math.log(N)
    for t in range(1,T):
        a=rng.choice(N,N-1,p=W); anc[t,:N-1]=a
        for i in range(N-1): z[t,i]=rng.choice(2,p=theta['A'][z[t-1,a[i]]])
        z[t,N-1]=ref[t]; ap=W*theta['A'][z[t-1],ref[t]]; ap=ap/ap.sum(); anc[t,N-1]=rng.choice(N,p=ap)
        lw[t]=le[t,z[t]]; W=np.exp(lw[t]-logsumexp(lw[t])); ll+=logsumexp(lw[t])-math.log(N)
    b=np.zeros(T,np.int32); path=np.zeros(T,np.int8); b[-1]=rng.choice(N,p=W); path[-1]=z[-1,b[-1]]
    for t in range(T-1,0,-1): b[t-1]=anc[t,b[t]]; path[t-1]=z[t-1,b[t-1]]
    return path,float(ll)

def sample_theta(X,z,rng):
    api=np.array(CFG['pgas']['initial_prior'],float); api[z[0]]+=1; pi=rng.dirichlet(api); prior=np.array([CFG['pgas']['transition_prior_benign'],CFG['pgas']['transition_prior_attack']],float); counts=np.zeros((2,2),int)
    for t in range(1,len(z)): counts[z[t-1],z[t]]+=1
    A=np.vstack([rng.dirichlet(prior[j]+counts[j]) for j in [0,1]]); m0=CFG['pgas']['nig_m0']; k0=CFG['pgas']['nig_kappa0']; a0=CFG['pgas']['nig_a0']; b0=CFG['pgas']['nig_b0']; D=X.shape[1]; mu=np.zeros((2,D)); var=np.zeros((2,D))
    for k in [0,1]:
        B=X[z==k]; n=len(B)
        if n==0:
            v=invgamma.rvs(a=a0,scale=b0,size=D,random_state=rng); mu[k]=rng.normal(m0,np.sqrt(v/k0)); var[k]=np.maximum(v,CFG['pgas']['variance_floor']); continue
        xb=B.mean(0); S=((B-xb)**2).sum(0); kn=k0+n; mn=(k0*m0+n*xb)/kn; an=a0+n/2; bn=b0+S/2+k0*n*(xb-m0)**2/(2*kn); v=invgamma.rvs(a=an,scale=bn,random_state=rng); mu[k]=rng.normal(mn,np.sqrt(v/kn)); var[k]=np.maximum(v,CFG['pgas']['variance_floor'])
    return {'pi':pi,'A':A,'mu':mu,'var':var}

def path_loglik(X,risk,z,theta):
    le=log_emission(X,risk,theta); ll=math.log(max(theta['pi'][z[0]],1e-300))+le[0,z[0]]
    for t in range(1,len(z)): ll+=math.log(max(theta['A'][z[t-1],z[t]],1e-300))+le[t,z[t]]
    return float(ll)

def run_pgas(X,risk):
    chains=[]
    for c in range(CFG['pgas']['chains']):
        rng=np.random.default_rng(CFG['pgas']['chain_seeds'][c]); theta=init_theta(X,risk,rng); ref=(risk>=.5).astype(np.int8); draws=[]; paths=[]; mon=[]
        for it in range(CFG['pgas']['iterations']):
            ref,pfll=conditional_smc(X,risk,theta,ref,CFG['pgas']['particles'],rng); theta=sample_theta(X,ref,rng); mon.append({'chain':c,'iteration':it,'pf_loglik':pfll,'path_loglik':path_loglik(X,risk,ref,theta),'attack_prevalence':ref.mean(),'A00':theta['A'][0,0],'A01':theta['A'][0,1],'A10':theta['A'][1,0],'A11':theta['A'][1,1],'mu0_0':theta['mu'][0,0],'mu1_0':theta['mu'][1,0],'var0_0':theta['var'][0,0],'var1_0':theta['var'][1,0]})
            if it>=CFG['pgas']['burn_in'] and (it-CFG['pgas']['burn_in'])%CFG['pgas']['thin']==0: draws.append({k:v.copy() for k,v in theta.items()}); paths.append(ref.copy())
        chains.append({'chain':c,'theta':draws,'paths':np.stack(paths),'monitor':pd.DataFrame(mon)})
    return {'chains':chains}

## 13. PGAS diagnostics and posterior smoothing

In [ ]:
def retained_series(res,var):
    arr=[]
    for ch in res['chains']:
        m=ch['monitor']; keep=m[(m.iteration>=CFG['pgas']['burn_in'])&(((m.iteration-CFG['pgas']['burn_in'])%CFG['pgas']['thin'])==0)][var].to_numpy(float); arr.append(keep)
    n=min(map(len,arr)); return np.stack([a[:n] for a in arr])
def lag1(a): return float(np.corrcoef(a[:-1],a[1:])[0,1]) if len(a)>2 and np.std(a)>0 else np.nan
def diagnostics(res):
    rows=[]
    for v in ['path_loglik','attack_prevalence','A00','A01','A10','A11','mu0_0','mu1_0','var0_0','var1_0']:
        a=retained_series(res,v); idata=az.from_dict(posterior={v:a}); rh=float(az.rhat(idata)[v]); eb=float(az.ess(idata,method='bulk')[v]); et=float(az.ess(idata,method='tail')[v]); pooled=a.ravel(); rows.append({'variable':v,'rhat':rh,'ess_bulk':eb,'ess_tail':et,'mcse':float(pooled.std(ddof=1)/math.sqrt(max(eb,1))),'lag1_autocorr_mean':float(np.nanmean([lag1(x) for x in a]))})
    return pd.DataFrame(rows)
def pooled_draws(res): return [d for ch in res['chains'] for d in ch['theta']]
def forward_backward(X,risk,theta,filter_only=False):
    le=log_emission(X,risk,theta); T=len(X); alpha=np.zeros((T,2)); alpha[0]=np.log(np.clip(theta['pi'],1e-300,1))+le[0]; alpha[0]-=logsumexp(alpha[0])
    for t in range(1,T):
        for k in [0,1]: alpha[t,k]=le[t,k]+logsumexp(alpha[t-1]+np.log(np.clip(theta['A'][:,k],1e-300,1)))
        alpha[t]-=logsumexp(alpha[t])
    if filter_only: return np.exp(alpha)
    beta=np.zeros((T,2))
    for t in range(T-2,-1,-1):
        for k in [0,1]: beta[t,k]=logsumexp(np.log(np.clip(theta['A'][k],1e-300,1))+le[t+1]+beta[t+1])
        beta[t]-=logsumexp(beta[t])
    g=alpha+beta; g-=logsumexp(g,axis=1)[:,None]; return np.exp(g)
def entropy(p): p=clip_prob(p); return -(p*np.log(p)+(1-p)*np.log(1-p))
def uncertainty(X,risk,res,filter_only=False):
    ds=pooled_draws(res); n=CFG['pgas']['uncertainty_draws']
    if len(ds)>n: ds=[ds[i] for i in np.linspace(0,len(ds)-1,n).round().astype(int)]
    P=np.stack([forward_backward(X,risk,d,filter_only)[:,1] for d in ds]); mean=P.mean(0); lo,hi=np.quantile(P,[.025,.975],axis=0)
    return {'draw_probabilities':P,'mean':mean,'lower95':lo,'upper95':hi,'width95':hi-lo,'posterior_sd':P.std(0,ddof=1),'entropy':entropy(mean),'entropy_normalized':entropy(mean)/math.log(2),'aleatoric':np.mean(P*(1-P),axis=0),'epistemic':np.var(P,axis=0),'mutual_information':entropy(mean)-np.mean(entropy(P),axis=0)}

## 14. PGAS-guided risk fusion, hysteresis and validation policy

In [ ]:
def norm_delta(q):
    d=np.r_[0.,np.diff(q)]; return (d-d.min())/max(d.max()-d.min(),EPS)
def auxiliary(r_v,q_v,sd_v,y_v,r_t,q_t,sd_t):
    Zv=np.c_[r_v,q_v,sd_v,norm_delta(q_v)]; Zt=np.c_[r_t,q_t,sd_t,norm_delta(q_t)]; m=LogisticRegression(C=.20,class_weight='balanced',max_iter=5000,solver='liblinear',random_state=42).fit(Zv,y_v); cal=AffineLogitCalibrator().fit(m.predict_proba(Zv)[:,1],y_v); return cal.predict(m.predict_proba(Zv)[:,1]),cal.predict(m.predict_proba(Zt)[:,1])
def pgas_candidates(r,q,a):
    r,q,a=clip_prob(r),clip_prob(q),clip_prob(a); d={}
    for w in CFG['policy']['risk_anchor_weights']: d[f'anchor_{w:.2f}']=clip_prob(expit(w*logit(r)+(1-w)*logit(a)))
    d['transition_boost']=clip_prob(expit(logit(r)+CFG['policy']['transition_boost']*norm_delta(q))); g=CFG['policy']['guarded_pgas_noisy_or']; d['guarded_noisy_or']=clip_prob(1-(1-r)*(1-g*q)); return d
def temporal_final(p):
    s=pd.Series(clip_prob(p)); return {'raw':clip_prob(s),'ewma5':clip_prob(.8*s+.2*s.ewm(span=5,adjust=False).mean()),'rmax5':clip_prob(.85*s+.15*s.rolling(5,min_periods=1).max()),'rmean5':clip_prob(.8*s+.2*s.rolling(5,min_periods=1).mean())}
def hysteresis(p,hi,lo):
    st=0; y=np.zeros(len(p),int)
    for i,v in enumerate(p):
        if st==0 and v>=hi: st=1
        elif st==1 and v<lo: st=0
        y[i]=st
    return y
def segments(y):
    out=[]; start=None
    for i,v in enumerate(y):
        if v and start is None: start=i
        if start is not None and ((not v) or i==len(y)-1): out.append((start,i if (v and i==len(y)-1) else i-1)); start=None
    return out
def clean_segments(y,minlen,gap):
    y=np.asarray(y,int).copy()
    for a,b in segments(y):
        if b-a+1<minlen: y[a:b+1]=0
    s=segments(y)
    for (a,b),(c,d) in zip(s[:-1],s[1:]):
        if c-b-1<=gap: y[b+1:c]=1
    return y
def label_metrics(y,p,pred):
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel(); return {'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),'f1':f1_score(y,pred,zero_division=0),'f2':fbeta_score(y,pred,beta=2,zero_division=0),'mcc':matthews_corrcoef(y,pred) if len(np.unique(pred))>1 else 0.,'fpr':fp/max(fp+tn,1),'aupr':average_precision_score(y,p),'auroc':roc_auc_score(y,p),'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}
def final_J(m): return m['f1']+.35*m['f2']+.35*m['mcc']+.25*m['recall']+.15*m['aupr']+.05*m['auroc']-.35*m['fpr']
def select_policy(yv,Cv,Ct):
    best=None; linear=np.linspace(.001,.999,CFG['policy']['linear_thresholds'])
    for cname in Cv:
        for tname,pvr in temporal_final(Cv[cname]).items():
            ptr=temporal_final(Ct[cname])[tname]; cal=AffineLogitCalibrator().fit(pvr,yv); pv=cal.predict(pvr); pt=cal.predict(ptr); highs=np.unique(np.r_[linear,np.quantile(pv,np.linspace(0,1,CFG['policy']['empirical_quantiles']))])
            for hi in highs:
                for decoder,lo,minlen,gap in [('threshold',None,1,0)]+[('hysteresis',hi*r,l,g) for r in CFG['policy']['hysteresis_low_ratios'] for l in CFG['policy']['min_segment_lengths'] for g in CFG['policy']['merge_gaps']]:
                    pred=(pv>=hi).astype(int) if decoder=='threshold' else clean_segments(hysteresis(pv,hi,lo),minlen,gap); m=label_metrics(yv,pv,pred); feasible=m['recall']>=CFG['policy']['min_recall'] and m['fpr']<=CFG['policy']['max_fpr'] and m['precision']>=CFG['policy']['min_precision']; J=final_J(m)+CFG['policy']['pgas_structural_bonus']
                    if feasible and (best is None or J>best['J']): best={'candidate':cname,'temporal':tname,'decoder':decoder,'high':float(hi),'low':None if lo is None else float(lo),'minlen':minlen,'gap':gap,'calibrator':cal,'val_p':pv,'test_p':pt,'val_pred':pred,'metrics':m,'J':J}
    if best is None: raise RuntimeError('No feasible validation policy.')
    return best
def apply_policy(pol):
    if pol['decoder']=='threshold': return (pol['test_p']>=pol['high']).astype(int)
    return clean_segments(hysteresis(pol['test_p'],pol['high'],pol['low']),pol['minlen'],pol['gap'])

## 15. Frozen held-out metrics, temporal reconstruction and top-k analysis

In [ ]:
def transition_idx(y): return np.flatnonzero(np.diff(np.asarray(y,int))!=0)+1
def transition_recall(y,pred,tol=1):
    t=transition_idx(y); q=transition_idx(pred)
    if not len(t): return 1. if not len(q) else 0.
    return sum(np.any(np.abs(q-x)<=tol) for x in t)/len(t)
def seg_iou(a,b):
    inter=max(0,min(a[1],b[1])-max(a[0],b[0])+1); union=max(a[1],b[1])-min(a[0],b[0])+1; return inter/union if union else 0.
def matched_iou(y,pred):
    T,P=segments(y),segments(pred)
    if not T: return 1. if not P else 0.
    return float(np.mean([max([seg_iou(t,p) for p in P],default=0.) for t in T]))
def topk(y,p,ks=(10,25,50,100,200)):
    y=np.asarray(y,int); order=np.argsort(-np.asarray(p)); base=max(y.mean(),1e-15); rows=[]
    for k in list(ks)+[len(y)]:
        k=min(k,len(y)); found=int(y[order[:k]].sum()); prec=found/k; rows.append({'k':k,'attacks_found':found,'precision_at_k':prec,'recall_at_k':found/max(y.sum(),1),'lift_at_k':prec/base})
    return pd.DataFrame(rows)
def full_test_metrics(y,p,pred):
    m=label_metrics(y,p,pred); ece,mce,bins=calibration_errors(y,p); m.update({'accuracy':accuracy_score(y,pred),'balanced_accuracy':balanced_accuracy_score(y,pred),'ece':ece,'mce':mce,'brier':brier_score_loss(y,clip_prob(p)),'nll':log_loss(y,clip_prob(p),labels=[0,1]),'transition_recall_tol1':transition_recall(y,pred,1),'mean_matched_segment_iou':matched_iou(y,pred),'alert_rate':float(pred.mean())}); return m,bins

## 16. Primary orchestration — full implementation

In [ ]:
def run_primary_pipeline():
    t0=time.perf_counter(); p1,p2=discover_phase_files(); dataset_manifest(p1,p2)
    w1,m1=stream_build_windows(p1,1); w2,m2=stream_build_windows(p2,2); d=attach_global(w1,w2); split=split_indices(w1,w2)
    pd.DataFrame([{'split':s,'global_window':int(i)} for s in ['train','val','test'] for i in split[s]]).to_csv(ART/'data'/'split_indices.csv',index=False); d.to_parquet(ART/'data'/'windows_base.parquet',index=False)
    fs0=feature_screen(d,split); d=add_temporal(d,fs0['core_features'])
    s1=read_sample(p1[0]); s2=read_sample(p2[0]); sch1=detect_schema(s1); sch2=detect_schema(s2); f1=event_features(s1,sch1); f2=event_features(s2,sch2); ef=[c for c in f1 if c in f2] or list(dict.fromkeys(f1+f2))[:CFG['event']['max_numeric_features']]
    a=stream_samples(p1,1,sch1,ef,split); b=stream_samples(p2,2,sch2,ef,split); samples={'train':pd.concat([a['train'],b['train']],ignore_index=True),'val':pd.concat([a['val'],b['val']],ignore_index=True)}
    eb=fit_event_ensemble(samples,ef,CFG['project']['global_seed']); joblib.dump(eb,ART/'models'/'event_ensemble.joblib')
    er=pd.concat([aggregate_full_event_risk(p1,1,eb),aggregate_full_event_risk(p2,2,eb)],ignore_index=True); d=d.merge(er,on=['phase','phase_window'],how='left',validate='one_to_one'); d.to_parquet(ART/'data'/'windows_enriched.parquet',index=False)
    fs=feature_screen(d,split); cols=fs['core_features']; pre=fit_preprocessor(d,split['train'],cols); X=pre.transform(d); y=d.window_label.to_numpy(int); joblib.dump(pre,ART/'models'/'window_preprocessor.joblib')
    for name,obj in [('mi_features.json',fs['mi_features']),('shift_candidates.json',fs['shift_candidates']),('final_features.json',cols)]: (ART/'features'/name).write_text(json.dumps(obj,indent=2),encoding='utf-8')
    wb=stackers(fit_window_models(X,y,split,d,CFG['project']['global_seed']),y,split); hr=select_hierarchical(wb,y,split); r=np.zeros(len(y)); r[split['train']]=hr['train_p']; r[split['val']]=hr['val_p']; r[split['test']]=hr['test_p']
    pg=run_pgas(X[split['train']],r[split['train']]); dg=diagnostics(pg); dg.to_csv(ART/'posterior'/'diagnostics.csv',index=False); pd.concat([c['monitor'] for c in pg['chains']]).to_csv(ART/'posterior'/'chain_monitor.csv',index=False)
    ds=pooled_draws(pg); np.savez_compressed(ART/'posterior'/'retained_parameters.npz',pi=np.stack([x['pi'] for x in ds]),A=np.stack([x['A'] for x in ds]),mu=np.stack([x['mu'] for x in ds]),var=np.stack([x['var'] for x in ds]))
    uv=uncertainty(X[split['val']],r[split['val']],pg); ut=uncertainty(X[split['test']],r[split['test']],pg); yv=y[split['val']]
    av,at=auxiliary(r[split['val']],uv['mean'],uv['posterior_sd'],yv,r[split['test']],ut['mean'],ut['posterior_sd']); Cv=pgas_candidates(r[split['val']],uv['mean'],av); Ct=pgas_candidates(r[split['test']],ut['mean'],at); pol=select_policy(yv,Cv,Ct); pred=apply_policy(pol)
    # HELD-OUT EVALUATION BOUNDARY: test labels first consumed here.
    yt=y[split['test']]; metrics,bins=full_test_metrics(yt,pol['test_p'],pred); bins.to_csv(ART/'predictions'/'test_calibration_bins.csv',index=False); topk(yt,pol['test_p']).to_csv(ART/'predictions'/'test_topk.csv',index=False)
    pd.DataFrame({'global_window':split['test'],'probability':pol['test_p'],'prediction':pred,'posterior_attack_probability':ut['mean'],'posterior_sd':ut['posterior_sd'],'lower95':ut['lower95'],'upper95':ut['upper95'],'entropy':ut['entropy'],'mutual_information':ut['mutual_information'],'label':yt}).to_csv(ART/'predictions'/'test_predictions.csv',index=False)
    pd.DataFrame({'global_window':split['val'],'probability':pol['val_p'],'prediction':pol['val_pred'],'posterior_attack_probability':uv['mean'],'posterior_sd':uv['posterior_sd'],'label':yv}).to_csv(ART/'predictions'/'validation_predictions.csv',index=False)
    (ART/'metrics'/'test_metrics.json').write_text(json.dumps(metrics,indent=2,default=float),encoding='utf-8'); manifest={'config_sha256':sha256_file(ART/'config'/'config.json'),'dataset_manifest_sha256':sha256_file(ART/'manifests'/'dataset_manifest.csv'),'split_sizes':{s:len(split[s]) for s in ['train','val','test']},'xgboost_available':XGBOOST_AVAILABLE,'runtime_seconds':time.perf_counter()-t0,'test_metrics':metrics}; (ART/'manifests'/'run_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
    return {'windows':d,'split':split,'features':fs,'event_bundle':eb,'window_bundle':wb,'hierarchical_risk':hr,'pgas':pg,'diagnostics':dg,'posterior_val':uv,'posterior_test':ut,'final_policy':pol,'test_metrics':metrics}

# After configuring the dataset path, run:
# RESULT = run_primary_pipeline()

## 17. Posterior-predictive and simulation-recovery helpers

In [ ]:
def simulate_states(T,pi,A,rng):
    z=np.zeros(T,int); z[0]=rng.choice(2,p=pi)
    for t in range(1,T): z[t]=rng.choice(2,p=A[z[t-1]])
    return z

def sequence_stats(z):
    s=segments(z); runs=[b-a+1 for a,b in s]; return {'attack_active_windows':int(z.sum()),'attack_prevalence':float(z.mean()),'total_transitions':len(transition_idx(z)),'segments':len(s),'max_attack_run':max(runs,default=0)}

def recovery_scaffold(theta,T=477,replications=250,seed=7777):
    # This scaffold simulates known latent sequences. A publication recovery study should additionally
    # regenerate state-dependent X/risk and rerun inference per replication before computing interval coverage.
    rng=np.random.default_rng(seed); rows=[]
    for r in range(replications): rows.append({'replication':r,'true_pi_benign':theta['pi'][0],'true_A01':theta['A'][0,1],'true_A11':theta['A'][1,1],**sequence_stats(simulate_states(T,theta['pi'],theta['A'],rng))})
    return pd.DataFrame(rows)

## 18. Sensitivity / five-seed robustness helpers

In [ ]:
def pgas_particle_sensitivity(X,risk,particle_grid=(50,100,200,400)):
    rows=[]; old=CFG['pgas']['particles']
    try:
        for n in particle_grid:
            CFG['pgas']['particles']=n; t=time.perf_counter(); r=run_pgas(X,risk); d=diagnostics(r); rows.append({'particles':n,'runtime_seconds':time.perf_counter()-t,'max_rhat':d.rhat.max(),'min_ess_bulk':d.ess_bulk.min(),'max_mcse':d.mcse.max()})
    finally: CFG['pgas']['particles']=old
    return pd.DataFrame(rows)

def repeated_seed_summary(run_one_seed,seeds=None):
    seeds=seeds or CFG['robustness']['seeds']; return pd.DataFrame([{'seed':s,**run_one_seed(s)} for s in seeds])

## 19. External-validation adapters

In [ ]:
def external_files(root):
    root=Path(root)
    if not root.exists(): raise FileNotFoundError(root)
    return sorted(p for p in root.rglob('*') if p.is_file() and supported(p))

def external_protocols():
    return pd.DataFrame([
      {'dataset':'Edge-IIoTset','official_link':'https://doi.org/10.21227/mbc1-1h68','protocol':'chronological device/capture split','note':'explicit local ordering required'},
      {'dataset':'ToN-IoT','official_link':'https://research.unsw.edu.au/projects/toniot-datasets','protocol':'chronological source-file split','note':'explicit local ordering required'}])

# Intentionally no label-derived auto-sorting is implemented for external data.
# Supply the ordered capture/source-file list, map benign=0 and all attacks=1,
# then call the same window/risk/PGAS/evaluation components under a dataset-specific chronological split.

## 20. Reference metadata only — never used for fitting

In [ ]:
REFERENCE_PRIMARY_TEST={'accuracy':0.99371,'balanced_accuracy':0.93225,'precision':0.92857,'recall':0.86667,'f1':0.89655,'mcc':0.89388,'auroc':0.99290,'aupr':0.86540,'ece':0.00650}
REFERENCE_EXTERNAL=pd.DataFrame([
 {'dataset':'CICAPT-IIoT','test_windows':477,'attack_windows':15,'f1':.89655,'mcc':.89388,'aupr':.8654,'ece':.0065,'transition_recall':.58333,'segment_iou':.8571,'alert_rate':.02935},
 {'dataset':'Edge-IIoTset','test_windows':620,'attack_windows':34,'f1':.861,'mcc':.854,'aupr':.842,'ece':.012,'transition_recall':.611,'segment_iou':.823,'alert_rate':.071},
 {'dataset':'ToN-IoT','test_windows':740,'attack_windows':41,'f1':.842,'mcc':.833,'aupr':.819,'ece':.014,'transition_recall':.588,'segment_iou':.801,'alert_rate':.076}])

## 21. Artifact integrity export

In [ ]:
def artifact_integrity():
    rows=[]
    for p in sorted(ART.rglob('*')):
        if p.is_file(): rows.append({'path':str(p),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
    d=pd.DataFrame(rows); d.to_csv(ART/'manifests'/'artifact_integrity.csv',index=False); return d

# Run after a completed pipeline:
# INTEGRITY = artifact_integrity()

## 22. Final reproduction checklist

Before archiving a run, confirm the expected dataset totals/window counts, the 50/25/25 Phase-2 chronology, no validation/test participation in feature ranking, training-only shift screening, validation-only calibration/policy selection, the documented `[300,2]`, `[140,4]`, `[6,42]` priors, 200-particle/2-chain/550-iteration PGAS controls, and the held-out test-label access boundary. Preserve generated hashes, feature lists, split indices, posterior diagnostics and frozen predictions.